In [ ]:
import os
import pandas as pd
from sklearn import datasets
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn import tree
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation


In [ ]:
init = pd.read_csv("C:\Punyaku\Rully\Botnet/init.csv")
init2 = pd.read_csv("C:\Punyaku\Rully\Botnet/init2.csv")
init3 = pd.read_csv("C:\Punyaku\Rully\Botnet/init3.csv")
init4 = pd.read_csv("C:\Punyaku\Rully\Botnet/init4.csv")
init5 = pd.read_csv("C:\Punyaku\Rully\Botnet/init5.csv")
dns1 = pd.read_csv("C:\Punyaku\Rully\Botnet/dns_application_2017_1.csv")
dns = pd.read_csv("C:\Punyaku\Rully\Botnet/dns_application_2017.csv")
appdns = pd.read_csv("C:\Punyaku\Rully\Botnet/applicationDNS.csv")

In [ ]:
df_botnet = pd.concat([init, init2, init3, init4, init5])
df_normal = pd.concat([dns1, dns, appdns])

# READ

In [ ]:
df_normal.head()

In [ ]:
df_botnet.head()

# Labelling

In [ ]:
df_normal['label'] = 'Normal'
df_botnet['label'] = 'Botnet'

# Filtering By Times

In [ ]:
df_normal['ts'] = pd.to_datetime(df_normal['ts'])

df_botnet['ts'] = df_botnet['ts'].str.replace('Z', '')  # Remove 'Z' from the timestamp string
try:
    df_botnet['ts'] = pd.to_datetime(df_botnet['ts'], format="%Y-%m-%dT%H:%M:%S")
except ValueError:
    df_botnet['ts'] = pd.to_datetime(df_botnet['ts'], format="%Y-%m-%dT%H:%M:%S.%f", errors='coerce')


In [ ]:
df_normal['time'] = df_normal['ts'].dt.strftime('%H:%M:%S')
df_normal['date'] = df_normal['ts'].dt.strftime('%Y-%m-%d')

df_botnet['time'] = df_botnet['ts'].dt.strftime('%H:%M:%S')
df_botnet['date'] = df_botnet['ts'].dt.strftime('%Y-%m-%d')

In [ ]:
df_normal.insert(2, 'time', df_normal.pop('time'))
df_normal.insert(3, 'date', df_normal.pop('date'))

df_botnet.insert(2, 'time', df_botnet.pop('time'))
df_botnet.insert(3, 'date', df_botnet.pop('date'))

In [ ]:
df_normal = df_normal.sort_values(['date', 'time'], ascending=True)

df_botnet = df_botnet.sort_values(['date', 'time'], ascending=True)

In [ ]:
df_normal.head()

In [ ]:
df_botnet.head()

# Drop Unused Column

In [ ]:
df_normal = df_normal.drop(['geo.orig.country_code','geo.orig.region','geo.orig.city',
              'geo.resp.latitude','geo.resp.longitude','geo.orig.latitude',
              'tunnel_parents','geo.orig.longitude','geo.resp.country_code',
              'geo.resp.region','geo.resp.city','local_orig','local_resp'], axis=1)


df_botnet = df_botnet.drop(['geo.orig.country_code','geo.orig.region','geo.orig.city',
              'geo.resp.latitude','geo.resp.longitude','geo.orig.latitude',
              'tunnel_parents','geo.orig.longitude','geo.resp.country_code',
              'geo.resp.region','geo.resp.city','local_orig','local_resp'], axis=1)

# Handling Missing Value

In [ ]:
#Replace With Mean
def replace_missing_with_mean(df, columns):
    #Count mean in column
    mean_values = df[columns].mean()
    
    # Replace missing value with mean
    df[columns] = df[columns].fillna(mean_values)
    
    return df

#Replace With Median
def replace_missing_with_median(df, columns):
    #Get Median in column
    median_values = df[columns].median()
    
    # Replace missing value with median
    df[columns] = df[columns].fillna(median_values)
    
    return df

#Replace with Mode
def replace_missing_with_mode(df, columns):
    #Get Mode Value in Column
    mode_values = df[columns].mode().iloc[0]
    
    # Replace missing value with median
    df[columns] = df[columns].fillna(mode_values)
    
    return df

#Replace with 0 Value
def replace_missing_with_zero(df, columns):
    #Zero Value
    zero_value = 0
    
    # Replace missing value with median
    df[columns] = df[columns].fillna(zero_value)
    
    return df

def replace_missing_with_previous_row(df, column):

    df[column].fillna(method='ffill', inplace=True)
    
    return df

## Drop Missing Value In Duration

In [ ]:
df_normal = df_normal.dropna(subset=['duration'])

df_botnet = df_botnet.dropna(subset=['duration'])

## Handling Missing Value With Previous Row

In [ ]:
df_normal = replace_missing_with_previous_row(df_normal, 'service')
df_normal = replace_missing_with_previous_row(df_normal, 'orig_bytes')
df_normal = replace_missing_with_previous_row(df_normal, 'resp_bytes')
df_normal = replace_missing_with_previous_row(df_normal, 'history')


df_botnet = replace_missing_with_previous_row(df_botnet, 'service')
df_botnet = replace_missing_with_previous_row(df_botnet, 'orig_bytes')
df_botnet = replace_missing_with_previous_row(df_botnet, 'resp_bytes')
df_botnet = replace_missing_with_previous_row(df_botnet, 'history')

In [ ]:
missing = pd.isnull(df_normal).sum()
missing

In [ ]:
df_botnet = replace_missing_with_previous_row(df_botnet, 'service')

In [ ]:
missing = pd.isnull(df_botnet).sum()
missing

In [ ]:
df_botnet = df_botnet.dropna(subset=['service'])

In [ ]:
pd.isnull(df_botnet).sum()

# Merge Dataframe Normal & Botnet

In [ ]:
df = pd.concat([df_botnet,df_normal])
df

# Ekstraksi Fitur

### Handling Duration Data

In [ ]:
short_values = [value for value in df['duration'] if len(value) < 5]
print(short_values)

In [ ]:
print(df['duration'])

In [ ]:
def process_time_string(time_string):
    time_unit = time_string[-2:]
    time_unit_s = time_string[-1:]
    time_value = time_string.replace('.', '')
    time_value_new = time_string.replace('.', '')
    time_value_new2 = time_string.replace('.', '')
    time_value_new3 = time_string.replace('.', '')
    if time_string.strip():
        if len(time_string) > 4:
            if time_string[1] == 'm':
                time_value = float(time_string[0]) * 60
                # time_value_new = time_value_new[2:]
                if time_unit == 'ms':
                    time_value_new = float(time_string[2:].replace('ms', '')) / 1000
                    time_total = time_value + time_value_new
                    return time_total
                elif time_unit == 'us':
                    time_value_new = float(time_string[2:].replace('us', '')) / 1000000
                    time_total = time_value + time_value_new
                    return time_total 
                elif time_unit_s == 's':
                    time_value_new = float(time_string[2:].replace('s', '')) 
                    time_total = time_value + time_value_new
                    return time_total
            elif time_string[2] == 'm':
                time_value = float(time_string[0]) * 60 * 10 + float(time_string[1]) * 60
                # time_value_new = time_string.replace('m', '')
                if time_unit == 'ms':
                    time_value_new = float(time_string[3:].replace('ms', '')) / 1000
                    time_total = time_value + time_value_new
                    return time_total
                elif time_unit == 'us':
                    time_value_new = float(time_string[3:].replace('us', '')) / 1000000
                    time_total = time_value + time_value_new
                    return time_total 
                elif time_unit_s == 's':
                        time_value_new = float(time_string[3:].replace('s', ''))
                        time_total = time_value + time_value_new
                        return time_total
            elif time_string[1] == 'd':
                time_value = float(time_string[0]) * 86400
                if time_string[3] == 'h':
                    time_value_new = float(time_string[2]) * 3600
                    if time_string[5] == 'm':
                        time_value_new2 = float(time_string[4]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[6:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total 
                        elif time_unit == 'us':
                            time_value_new3 = float(time_string[6:].replace('us', '')) / 1000000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total 
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[6:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                    elif time_string[6] == 'm':
                        time_value_new2 = float(time_string[4]) * 60 * 10 + float(time_string[5]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[7:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                        elif time_unit == 'us':
                            time_value_new3 = float(time_string[7:].replace('us', '')) / 1000000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total 
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[7:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                elif time_string[4] == 'h':
                    time_value_new = float(time_string[2]) * 3600 * 10 + float(time_string[3]) * 3600
                    if time_string[6] == 'm':
                        time_value_new2 = float(time_string[5]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[7:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                        elif time_unit == 'us':
                            time_value_new3 = float(time_string[7:].replace('us', '')) / 1000000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[7:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                    elif time_string[7] == 'm':
                        time_value_new2 = float(time_string[5]) * 60 * 10 + float(time_string[6]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[8:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                        elif time_unit == 'us':
                            time_value_new3 = float(time_string[8:].replace('us', '')) / 1000000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[8:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
            elif time_string[2] == 'd':
                time_value = float(time_string[0]) * 86400 * 10 + float(time_string[1]) *86400
                if time_string[4] == 'h':
                    time_value_new = float(time_string[3]) * 3600
                    if time_string[6] == 'm':
                        time_value_new2 = float(time_string[5]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[7:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total 
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[7:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                    elif time_string[7] == 'm':
                        time_value_new2 = float(time_string[5]) * 60 * 10 + float(time_string[6]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[8:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total 
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[8:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                elif time_string[5] == 'h':
                    time_value_new = float(time_string[3]) * 3600 * 10 + float(time_string[4]) * 3600
                    if time_string[6] == 'm':
                        time_value_new2 = float(time_string[5]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[7:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total 
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[7:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
                    elif time_string[7] == 'm':
                        time_value_new2 = float(time_string[5]) * 60 * 10 + float(time_string[6]) * 60
                        if time_unit == 'ms':
                            time_value_new3 = float(time_string[7:].replace('ms', '')) / 1000
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total 
                        elif time_unit_s == 's':
                            time_value_new3 = float(time_string[7:].replace('s', '')) 
                            time_total = time_value + time_value_new + time_value_new2 + time_value_new3
                            return time_total
            elif time_unit == 'ms':
                time_value = float(time_string.replace('ms', '')) / 1000
                return time_value
            elif time_unit == 'us':
                time_value = float(time_string.replace('us', '')) / 1000000
                return time_value
            elif time_unit_s == 's':
                time_value = float(time_string.replace('s', ''))
                return time_value
            else:
                return time_value
        else: 
            if time_unit == 'ms':
                time_value = float(time_string.replace('ms', '')) / 1000
                return time_value
            elif time_string[1] == 'm':
                time_value = float(time_string[0]) * 60
                # time_value_new = time_value_new[2:]
                if time_unit == 'ms':
                    time_value_new = float(time_string[2:].replace('ms', '')) / 1000
                    time_total = time_value + time_value_new
                    return time_total
                elif time_unit == 'us':
                    time_value_new = float(time_string[2:].replace('us', '')) / 1000000
                    time_total = time_value + time_value_new
                    return time_total 
                elif time_unit_s == 's':
                    time_value_new = float(time_string[2:].replace('s', '')) 
                    time_total = time_value + time_value_new
                    return time_total
            elif time_unit == 'us':
                time_value = float(time_string.replace('us', '')) / 1000000
                return time_value
            elif time_unit_s == 's':
                time_value = float(time_string.replace('s', ''))
                return time_value
            else:
                return time_value
            

df['duration'] = df['duration'].apply(lambda x: process_time_string(x) if isinstance(x, str) else x)

print(df['duration'])

In [ ]:
pd.isnull(df).sum()

### Ekstraksi

In [ ]:
df['PX'] = df['orig_pkts'] + df['resp_pkts']

df['NNP'] = np.where(df['orig_pkts'] + df['resp_pkts'] == 0, 1, 0)

df['NSP'] = ((63 >= df['orig_pkts'] + df['resp_pkts']) & (df['orig_pkts'] + df['resp_pkts'] <= 400)).astype(int)

df['PSP'] = df['NSP'] / df['PX'] 

df['IOPR'] = df['orig_pkts'] + df['resp_pkts']

df['reconnection'] = np.where(df['history'].str.contains('Sr%'), 1, 0)
df['reconnection'] = df['reconnection'].fillna(0).astype(int)

df['FPS'] = df['orig_ip_bytes'] / df['orig_pkts']
df['FPS'] = df['FPS'].fillna(0)

df['TBT'] = df['orig_ip_bytes'] + df['resp_ip_bytes']

df['APL'] = (df['orig_ip_bytes'] + df['resp_ip_bytes']) / df['PX']

df['BS'] = ((df['orig_ip_bytes']+ df['resp_ip_bytes'])* 8 ) / df['duration']

df['PS'] =(df['orig_pkts'] + df['resp_pkts']) / df['duration'] 

df['PPS'] = (df['orig_pkts'] + df['resp_pkts']) / df['duration'] 

# Normalisasi Data

In [ ]:
# Normalisasi
sc = MinMaxScaler(feature_range=(0, 1))

#Reset Index untuk duplikasi
dfnew = df
df = df.reset_index(drop=True)

# Apply the Min-Max Scaler to the numerical columns
dfnew = pd.DataFrame(data=sc.fit_transform(df.select_dtypes(include=['float64', 'int64'])), columns=df.select_dtypes(include=['float64', 'int64']).columns)

#duplikasi label
dfnew['label'] = df['label']


In [ ]:
pd.isnull(dfnew).sum()

# Shuffle Row

In [ ]:
dfnew = dfnew.sample(frac=1).reset_index(drop=True)
dfnew

# Klasifikasi Decision Tree

### Data Keseluruhan

In [ ]:
from sklearn.model_selection import train_test_split
X = dfnew.drop('label', axis=1)
y = dfnew['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
X

### Data Balanced

In [ ]:
# Memotong jumlah data botnet agar sama dengan jumlah data normal
botnet_count = (dfnew['label'] == 'Botnet').sum()
normal_count = (dfnew['label'] == 'Normal').sum()

if botnet_count > normal_count:
    data1 = pd.concat([dfnew[dfnew['label'] == 'Botnet'].head(normal_count), dfnew[dfnew['label'] == 'Normal']])
else:
    data1 = pd.concat([dfnew[dfnew['label'] == 'Botnet'], dfnew[dfnew['label'] == 'Normal'].head(botnet_count)])

# Menampilkan jumlah data di setiap kategori setelah dipotong
print(data1['label'].value_counts())

In [ ]:
y1 = data1['label']
X1 = data1.drop('label', axis=1)

In [ ]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, stratify=y1)

# Klasifikasi

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from numpy import mean
from numpy import std
from sklearn.tree import export_text

# Decision Tree Entire Data

In [ ]:
# Decision Tree classifer object
clf = DecisionTreeClassifier(max_depth = None, min_samples_split = 2)

# Train Decision Tree
clf = clf.fit(X_train,y_train)

#Predict
y_pred = clf.predict(X_test)

# hasil
score = metrics.accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(score)
print(report)

# K-Fold Decision Tree Entire Data

In [ ]:
cv = KFold(n_splits=10, random_state=1, shuffle=True)
scores = cross_val_score(clf, X, y, scoring='accuracy', cv=cv, n_jobs=-1)
print('Accuracy: %.3f (%.3f)' % (mean(scores), std(scores)))

# Decision Tree Balanced Data

In [ ]:
# Decision Tree classifer object
model1 = DecisionTreeClassifier(max_depth = None, min_samples_split = 2)

# Train Decision Tree
model1 = model1.fit(X_train1,y_train1)

#Predict
y_pred1 = model1.predict(X_test1)

# hasil
score = metrics.accuracy_score(y_test1, y_pred1)
report = classification_report(y_test1, y_pred1)
print(score)
print(report)

# K-Fold Decision Tree Balanced Data

In [ ]:
cv2 = KFold(n_splits=10, random_state=1, shuffle=True)
scores = cross_val_score(model1, X, y, scoring='accuracy', cv=cv2, n_jobs=-1)
print('Accuracy: %.3f (%.3f)' % (mean(scores), std(scores)))

# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test1, y_pred1)

print("Confusion Matrix:")
print(cm)

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [ ]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size=0.2, stratify=y1)

In [ ]:
X_train1.info()

In [ ]:
rf = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)

In [ ]:
rf = rf.fit(X_train1,y_train1)

In [ ]:
y_pred1 = rf.predict(X_test1)

In [ ]:
print("Classification Report:")
print(classification_report(y_test1, y_pred1))

## Confusion Matrix Random Forest

In [ ]:
# from sklearn.metrics import plot_confusion_matrix
from sklearn.metrics import confusion_matrix

matrix = confusion_matrix(y_test1, y_pred1)
plt.show() 

# K-Fold Random Forest

In [ ]:
kfrf = KFold(n_splits=10, random_state=1, shuffle=True)
scores = cross_val_score(rf, X_train1, y_train1, scoring='accuracy', cv=kfrf, n_jobs=-1)
print('Accuracy: %.3f (%.3f)' % (mean(scores), std(scores)))

# SVM

In [ ]:
# from sklearn import svm
# from sklearn.metrics import accuracy_score

# svm_model = svm.SVC()
# svm_model.fit(X_train1, y_train1)

# y_pred1 = svm_model.predict(X_test1)
# accuracy = accuracy_score(y_test1, y_pred1)
# print("Accuracy:", accuracy)

## Confusion Matrix SVM

In [ ]:
# from sklearn.metrics import plot_confusion_matrix
# matrix = confusion_matrix(y_test1, y_pred1)
# plot_confusion_matrix(svm_model, X_test1, y_test1)  
# plt.show() 

## K-Fold Cross Validation

In [ ]:
# k = 5
# cv_scores = cross_val_score(svm_model, X_train1, y_train1, cv=k)

# # Calculate the mean and standard deviation of the cross-validation scores
# mean_accuracy = cv_scores.mean()
# std_accuracy = cv_scores.std()

# print(f"Cross-Validation Accuracy (k={k}): {mean_accuracy:.4f} +/- {std_accuracy:.4f}")

# Neural Network

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Label encoding for binary classification (Botnet = 0, Normal = 1)
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train1)
y_test_encoded = label_encoder.transform(y_test1)

# Create a sequential model
ann = Sequential()

# Add a fully connected layer with 32 units and 'relu' activation function
ann.add(Dense(32, activation='relu', input_dim=X_train1.shape[1]))

# Add another fully connected layer with 16 units and 'relu' activation function
ann.add(Dense(16, activation='relu'))

# Add the output layer with 1 unit and sigmoid activation function (for binary classification)
ann.add(Dense(1, activation='sigmoid'))

# Compile the model with binary cross-entropy loss and Adam optimizer
ann.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
epochs = 10
batch_size = 32
ann.fit(X_train1, y_train_encoded, epochs=epochs, batch_size=batch_size, validation_split=0.1)

# Evaluate the model on the test set
loss, accuracy = ann.evaluate(X_test1, y_test_encoded)
print(f"Test loss: {loss:.4f}, Test accuracy: {accuracy:.4f}")


# Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate predictions for the test data
y_pred_proba = ann.predict(X_test1)
y_pred = (y_pred_proba > 0.5).astype(int)

# Create a confusion matrix
cm = confusion_matrix(y_test_encoded, y_pred)

class_names = label_encoder.inverse_transform([0, 1])

# Plot the confusion matrix using seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()


# Cross Validation ANN

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier

# Define the number of folds (k)
k = 9

# Define a function to create your ANN model
def create_ann():
    ann = Sequential()
    ann.add(Dense(32, activation='relu', input_dim=X_train1.shape[1]))
    ann.add(Dense(16, activation='relu'))
    ann.add(Dense(1, activation='sigmoid'))
    ann.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return ann

# Create a KerasClassifier wrapper
estimator = KerasClassifier(build_fn=create_ann, epochs=10, batch_size=32, verbose=0)

# Create a pipeline with a standard scaler and the KerasClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # You can choose to scale the features if needed
    ('keras', estimator)
])

# Create a stratified k-fold cross-validator
cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

# Perform cross-validation and get predicted labels
predicted_labels = cross_val_predict(pipeline, X_train1, y_train_encoded, cv=cv, method='predict')

# Calculate accuracy for each fold
fold_accuracies = [np.mean(predicted_labels[i] == y_train_encoded[i]) for i in range(k)]

# Print accuracy for each fold
for fold_num, accuracy in enumerate(fold_accuracies, start=1):
    print(f"Fold {fold_num} Accuracy: {accuracy:.4f}")

# Calculate the mean and standard deviation of the cross-validation accuracies
mean_accuracy = np.mean(fold_accuracies)
std_accuracy = np.std(fold_accuracies)

print(f"\nCross-Validation Accuracy (k={k}): {mean_accuracy:.4f} +/- {std_accuracy:.4f}")
